# K-medoids and CLARA lab

Compare a medoid-based partition with a sampled CLARA search on a reproducible synthetic spatial point pattern. The coordinates are illustrative planar units, not a geographic dataset.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/k-medoids'))
from k_medoids import euclidean_dissimilarity, pam, clara, clarans


## 1. Build two separated groups


In [ ]:
rng = np.random.default_rng(23)
a = rng.normal(loc=(2, 2), scale=0.45, size=(35, 2))
b = rng.normal(loc=(8, 7), scale=0.55, size=(35, 2))
points = np.vstack([a, b])
D = euclidean_dissimilarity(points)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(points[:, 0], points[:, 1], s=18)
ax.set(title='Synthetic point pattern', xlabel='x', ylabel='y')
ax.set_aspect('equal')


## 2. Fit PAM and inspect observed medoids


In [ ]:
fit = pam(D, k=2)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(points[:, 0], points[:, 1], c=fit['labels'], s=22)
ax.scatter(points[fit['medoids'], 0], points[fit['medoids'], 1], marker='X', s=180, edgecolor='black')
ax.set(title=f'PAM k-medoids — objective {fit["cost"]:.2f}', xlabel='x', ylabel='y')
ax.set_aspect('equal')
print('Medoid row indices:', fit['medoids'])
print('Swap iterations:', fit['iterations'], 'converged:', fit['converged'])


## 3. Compare CLARA sampling

CLARA fits PAM on repeated samples and evaluates each medoid set on the full dataset. Change sample size and the random seed to see how sampling affects the objective.


In [ ]:
approx = clara(D, k=2, n_samples=10, sample_size=25, random_state=17)
randomized = clarans(D, k=2, max_neighbors=40, num_local=5, random_state=17)
print('PAM objective:', fit['cost'])
print('CLARA full-data objective:', approx['cost'])
print('CLARA medoid indices:', approx['medoids'])
print('Sample used:', approx['sample'])
print('CLARANS randomized-search objective:', randomized['cost'])


## Investigate

1. Why must every medoid be one of the observed objects?
2. How does PAM differ from k-means when clusters contain outliers?
3. What does CLARA gain from sampling, and what does it lose?
4. Why does this implementation still need the full distance matrix?
5. Which extra constraints would be needed for geographically contiguous or equal-size regions?
